In [0]:
# =====================================================================
# CELL 1: PLATFORM CHECK -> which AI models and Vector Search can we use?
# =====================================================================
# Same habit as the egress test: check what the platform offers BEFORE designing.
# Serving endpoints = models we can call: LLMs for the agents, embedding models for search.
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()            # talks to this workspace with YOUR permissions

names = sorted(e.name for e in w.serving_endpoints.list())
print(f"{len(names)} serving endpoints available:")
for n in names:
    print("  ", n)

# Vector Search = Databricks' managed search index for embeddings
try:
    eps = list(w.vector_search_endpoints.list_endpoints())
    print("\n✅ Vector Search API reachable. Existing endpoints:", [e.name for e in eps])
except Exception as error:
    print("\n❌ Vector Search not available:", type(error).__name__, str(error)[:200])

In [0]:
# =====================================================================
# CELL 2: Download plain text of selected reports -> BRONZE Volume
# =====================================================================
# * PARALLEL: 8 downloads at a time (ThreadPoolExecutor) -> ~8x faster
# * INCREMENTAL: skip files already downloaded -> re-running is cheap (idempotent)
import os, requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=1.5,
                                                        status_forcelist=[429, 500, 502, 503, 504])))
session.mount("http://",  HTTPAdapter(max_retries=Retry(total=3, backoff_factor=1.5)))

TEXT_DIR = "/Volumes/procintel/bronze/raw_files/doc_text"
os.makedirs(TEXT_DIR, exist_ok=True)

# Selection: ALL ICR / IEG / PAD + the 2 most recent ISRs per project
docs = spark.sql("""
  SELECT doc_id, project_id, project_name, doc_family, authority, doc_type, title, doc_date, txturl
  FROM procintel.gold.rag_documents
  WHERE doc_family IN ('ICR','IEG_ICRR','IEG_PPAR','PAD')
  UNION ALL
  SELECT doc_id, project_id, project_name, doc_family, authority, doc_type, title, doc_date, txturl
  FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY project_id ORDER BY doc_date DESC) AS rn
        FROM procintel.gold.rag_documents WHERE doc_family = 'ISR')
  WHERE rn <= 2
""").toPandas()
print("documents selected:", len(docs), "|", docs.doc_family.value_counts().to_dict())

def fetch(row):
    """Download one document's text. Returns (doc_id, status)."""
    path = f"{TEXT_DIR}/{row.doc_id}.txt"
    if os.path.exists(path):
        return row.doc_id, "cached"                       # already have it -> skip
    try:
        r = session.get(row.txturl, timeout=60)
        if r.status_code != 200:
            return row.doc_id, f"http_{r.status_code}"
        text = r.content.decode("utf-8", errors="replace")  # bad bytes -> '�' instead of crashing
        if len(text) < 500:
            return row.doc_id, "too_short"                  # empty or placeholder file
        with open(path, "w") as f:
            f.write(text)
        return row.doc_id, "ok"
    except Exception as error:
        return row.doc_id, f"error_{type(error).__name__}"

with ThreadPoolExecutor(max_workers=8) as pool:            # 8 downloads in parallel
    results = list(pool.map(fetch, docs.itertuples()))

status = pd.Series(dict(results))
print(status.value_counts().to_dict())

In [0]:
# =====================================================================
# CELL 3: STRUCTURE-AWARE CHUNKING -> gold.doc_chunks
# =====================================================================
import re

HEADING = re.compile(r"^\s*(\d{1,2}(\.\d{1,2}){0,3}\.?\s+[A-Z][^\n]{2,90}|[A-Z][A-Z &,\-/]{6,80})\s*$")
#   matches "4.2 Procurement arrangements"  or  "LESSONS LEARNED"  (short numbered or ALL-CAPS line)
MAX_CHARS, MIN_CHARS = 1500, 200

def chunk_document(text):
    """Walk line by line. A heading closes the current chunk and starts a new section.
    Chunks never cross a heading and never exceed ~MAX_CHARS."""
    chunks, buf, section = [], [], "Front matter"
    def flush():
        body = " ".join(buf).strip()
        if len(body) >= MIN_CHARS:
            chunks.append((section, body))
        buf.clear()
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        if len(line) < 100 and HEADING.match(line):
            flush()
            section = line[:120]
            continue
        buf.append(line)
        if sum(len(x) for x in buf) > MAX_CHARS:
            flush()
    flush()
    return chunks

rows = []
for d in docs.itertuples():
    path = f"{TEXT_DIR}/{d.doc_id}.txt"
    if not os.path.exists(path):
        continue
    with open(path) as f:
        text = f.read()
    for i, (section, body) in enumerate(chunk_document(text)):
        rows.append({
            "chunk_id": f"{d.doc_id}_{i:04d}",
            "doc_id": d.doc_id, "project_id": d.project_id, "project_name": d.project_name,
            "doc_family": d.doc_family, "authority": float(d.authority),
            "title": d.title, "doc_date": d.doc_date, "section": section,
            "is_lesson": bool(re.search(r"LESSON", section, re.I)),
            "text": body})

chunks = pd.DataFrame(rows)
(spark.createDataFrame(chunks)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("procintel.gold.doc_chunks"))

# CHANGE DATA FEED: Delta records every row change -> a Vector Search index can
# sync INCREMENTALLY (only new/changed chunks get re-embedded).
spark.sql("ALTER TABLE procintel.gold.doc_chunks SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql("COMMENT ON TABLE procintel.gold.doc_chunks IS 'Structure-aware chunks of ICR/IEG/PAD/ISR reports, with section and authority, for RAG.'")

print("chunks:", len(chunks), "| documents:", chunks.doc_id.nunique())
print("by family:", chunks.doc_family.value_counts().to_dict())
print("lesson chunks:", int(chunks.is_lesson.sum()))
print("median chunk length:", int(chunks.text.str.len().median()), "chars")

In [0]:
# =====================================================================
# CELL 4: CURATE what goes into the search index -> gold.rag_chunks
# =====================================================================
# Keep ALL chunks from outcome documents (ICR, IEG, ISR).
# From PADs (design documents, 65% of chunks) keep only procurement, risk,
# implementation and lesson passages, so PAD volume can't drown out the lessons.
# RLIKE '(?i)...' = regex match, (?i) = ignore upper/lower case.
spark.sql("""
CREATE OR REPLACE TABLE procintel.gold.rag_chunks
TBLPROPERTIES (delta.enableChangeDataFeed = true)
COMMENT 'Curated chunks for Vector Search: all ICR/IEG/ISR + procurement/risk/lesson parts of PADs.'
AS SELECT * FROM procintel.gold.doc_chunks
WHERE doc_family <> 'PAD'
   OR is_lesson
   OR section RLIKE '(?i)procure|risk|implementation arrangement|lesson|fiduciary'
""")
print(spark.sql("""SELECT doc_family, COUNT(*) AS n FROM procintel.gold.rag_chunks
                   GROUP BY ALL ORDER BY n DESC""").toPandas())

In [0]:
# =====================================================================
# CELL 5: install the Vector Search Python client, then restart Python
# =====================================================================
%pip install -q databricks-vectorsearch
dbutils.library.restartPython()   # new packages need a fresh Python process

In [0]:
# =====================================================================
# CELL 6: FOCUSED corpus + Vector Search index (async, idempotent)
# =====================================================================
# DESIGN DECISION (documented): a first attempt indexed all 29k curated chunks.
# Free Edition's embedding endpoint runs ~60 chunks/min -> 6+ hours, so we index a
# FOCUSED corpus (~3k high-value passages). Scale-up path: provisioned-throughput
# embedding endpoint on a paid workspace -> index gold.rag_chunks with the same code.
#
# Keep:  ICR / IEG -> lessons + procurement / risk / implementation / outcome sections
#        PAD       -> lessons only (design docs = intentions, not outcomes)
#        ISR       -> passages that mention procurement
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)
ENDPOINT, INDEX = "procintel_vs", "procintel.ml.rag_focus_index"

# MERGE = UPSERT: insert new chunks, update changed ones, delete removed ones.
# Only CHANGED rows enter the Change Data Feed -> the index re-embeds only those.
FOCUS_FILTER = """
  (doc_family IN ('ICR','IEG_ICRR','IEG_PPAR')
     AND (is_lesson OR section RLIKE '(?i)procure|lesson|risk|implementation|outcome|factors|efficiency'))
  OR (doc_family = 'PAD' AND is_lesson)
  OR (doc_family = 'ISR' AND text RLIKE '(?i)procure|tender|bidding|contract')"""

spark.sql(f"""CREATE TABLE IF NOT EXISTS procintel.gold.rag_focus_chunks
              TBLPROPERTIES (delta.enableChangeDataFeed = true)
              AS SELECT * FROM procintel.gold.rag_chunks WHERE {FOCUS_FILTER}""")

spark.sql(f"""
MERGE INTO procintel.gold.rag_focus_chunks AS t
USING (SELECT * FROM procintel.gold.rag_chunks WHERE {FOCUS_FILTER}) AS s
ON t.chunk_id = s.chunk_id
WHEN MATCHED AND t.text <> s.text THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE""")
print("focused chunks:", spark.table("procintel.gold.rag_focus_chunks").count())


# Endpoint: create only if missing
if ENDPOINT not in [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]:
    vsc.create_endpoint_and_wait(name=ENDPOINT, endpoint_type="STANDARD")

# Index: create only if missing, otherwise just trigger a SYNC (picks up changed rows via CDF)
existing = [i["name"] for i in vsc.list_indexes(ENDPOINT).get("vector_indexes", [])]
if INDEX not in existing:
    vsc.create_delta_sync_index(                     # async: returns immediately
        endpoint_name=ENDPOINT, index_name=INDEX,
        source_table_name="procintel.gold.rag_focus_chunks",
        primary_key="chunk_id", pipeline_type="TRIGGERED",
        embedding_source_column="text",
        embedding_model_endpoint_name="databricks-gte-large-en",
        columns_to_sync=["chunk_id", "doc_id", "project_id", "project_name", "doc_family",
                         "authority", "title", "doc_date", "section", "is_lesson", "text"])
    print("🚀 index creation started (runs in background)")
else:
    vsc.get_index(ENDPOINT, INDEX).sync()            # incremental refresh
    print("🔄 index exists -> sync triggered")

In [0]:
# =====================================================================
# STATUS CHECK -> where exactly is the Vector Search build?
# =====================================================================
# Endpoint states: PROVISIONING -> ONLINE
# Index states:    PROVISIONING... -> ONLINE (indexed_row_count tells progress)
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

ep = vsc.get_endpoint("procintel_vs")
print("ENDPOINT state:", ep.get("endpoint_status", {}).get("state"),
      "|", ep.get("endpoint_status", {}).get("message", ""))

try:
    idx = vsc.get_index("procintel_vs", "procintel.ml.rag_focus_index").describe()
    st = idx.get("status", {})
    print("INDEX ready:", st.get("ready"),
          "| state:", st.get("detailed_state"),
          "| rows indexed:", st.get("indexed_row_count"),
          "\nmessage:", str(st.get("message", ""))[:400])
except Exception as error:
    print("INDEX: not created yet ->", type(error).__name__, str(error)[:200])

print("rows in source table:", spark.table("procintel.gold.rag_chunks").count())

In [0]:
# =====================================================================
# CELL 7: TEST -> hybrid search with a metadata filter
# =====================================================================
# query_type="HYBRID" = keyword match + meaning match, fused into one ranking.
# filters = only search inside outcome documents (where lessons live).
index = vsc.get_index(endpoint_name="procintel_vs", index_name="procintel.ml.rag_focus_index")

res = index.similarity_search(
    query_text="Why were civil works contracts delayed? land acquisition and site handover problems",
    columns=["doc_family", "project_id", "title", "section", "text"],
    filters={"doc_family": ["ICR", "IEG_ICRR", "IEG_PPAR"]},
    query_type="HYBRID",
    num_results=5)

for row in res["result"]["data_array"]:
    family, project, title, section, text, score = row
    print(f"[{family}] {project} | {section[:60]}\n   score={score:.3f} | {text[:200]}...\n")